# 🎓 Student Performance Analysis & Score Prediction

## Module 06 – Final Data Science Project

### Project Goal
Build an end-to-end Data Science project that analyzes student performance
and uses Machine Learning to predict the final mathematics score (`G3`).

### Workflow
**Data Collection → Data Cleaning → EDA → Visualization → Feature Engineering → ML → Evaluation → Insights**

### Tools
Python | Pandas | NumPy | Matplotlib | Seaborn | Scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Data Collection

The dataset is the UCI Student Performance dataset for mathematics (`student-mat.csv`).
It contains demographic, social, study-related and academic information about students.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

import io, zipfile, requests

response = requests.get(url, timeout=30)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("student-mat.csv") as f:
        df = pd.read_csv(f, sep=";")

print("Dataset shape:", df.shape)
display(df.head())

## 2. Data Understanding

Inspect columns, data types, missing values and summary statistics.


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumerical summary:")
display(df.describe().T)

## 3. Data Cleaning

The dataset is checked for missing and duplicate records. The target variable is the final mathematics grade `G3`.


In [ ]:
df = df.drop_duplicates().copy()

print("Shape after duplicate removal:", df.shape)
print("Total missing values:", int(df.isnull().sum().sum()))
print("Target statistics:")
display(df["G3"].describe().to_frame("G3"))

## 4. Exploratory Data Analysis

### 4.1 Distribution of Final Scores


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["G3"], bins=11, kde=True)
plt.title("Distribution of Final Mathematics Scores")
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="sex", y="G3")
plt.title("Final Score by Gender")
plt.xlabel("Gender")
plt.ylabel("Final Grade (G3)")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="studytime", y="G3")
plt.title("Final Score by Study Time")
plt.xlabel("Study Time Category")
plt.ylabel("Final Grade (G3)")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="G1", y="G3", alpha=0.7)
plt.title("First Period Grade vs Final Grade")
plt.xlabel("First Period Grade (G1)")
plt.ylabel("Final Grade (G3)")
plt.show()

## 5. Feature Relationships

Correlation is examined for numerical variables to identify relationships with the final score.


In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

plt.figure(figsize=(12,9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

print("Top numerical correlations with G3:")
display(corr["G3"].sort_values(ascending=False).to_frame("Correlation with G3").head(10))

## 6. Feature Preparation

Categorical columns are one-hot encoded and numerical columns are standardized where appropriate. The target `G3` is separated from the predictors.


In [ ]:
X = df.drop(columns=["G3"])
y = df["G3"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])
print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numeric_cols))

## 7. Model 1 – Linear Regression


In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("MAE :", round(linear_mae, 3))
print("RMSE:", round(linear_rmse, 3))
print("R²  :", round(linear_r2, 3))

## 8. Model 2 – Random Forest Regression


In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Regression")
print("MAE :", round(rf_mae, 3))
print("RMSE:", round(rf_rmse, 3))
print("R²  :", round(rf_r2, 3))

## 9. Model Comparison


In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [linear_mae, rf_mae],
    "RMSE": [linear_rmse, rf_rmse],
    "R2": [linear_r2, rf_r2]
}).round(3)

display(results)

best_model = results.loc[results["R2"].idxmax(), "Model"]
print("Best model based on R²:", best_model)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test, rf_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linewidth=2)
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title("Random Forest – Actual vs Predicted Scores")
plt.show()

## 10. Feature Importance – Random Forest


In [ ]:
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_estimator = rf_model.named_steps["model"]

feature_names = rf_preprocessor.get_feature_names_out()
importance = pd.Series(
    rf_estimator.feature_importances_,
    index=feature_names
).sort_values(ascending=False).head(15)

plt.figure(figsize=(9,6))
importance.sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.show()

display(importance.to_frame("Importance"))

## 11. Final Insights

- The project demonstrates a complete Data Science workflow from raw data to a trained ML model.
- Student performance can be explored through demographic, social, study and previous-grade variables.
- `G1` and other academic variables can provide strong predictive information for `G3`.
- Model performance should be judged using multiple regression metrics rather than a single number.
- The Random Forest model can capture non-linear relationships, while Linear Regression provides a simple baseline.

### Portfolio Note
This notebook is intentionally organized as an end-to-end project so it can be uploaded to GitHub
as a portfolio project.

## 12. Conclusion

The final project successfully combines data analysis, visualization and machine learning to study student performance and predict final mathematics scores. The workflow can be extended with hyperparameter tuning, cross-validation, feature selection and deployment as a web application.
